# 02 — VisA GPU experiments: sweep, follow-up, autoencoder, anomalib, MPDD, robustness

The experiments that need more than the 4 GB laptop: the ablation screen around the frozen
recipe (K1), the Gate E follow-up at three seeds (E8, pre-registered), the autoencoder rung
(Tier 1), our PatchCore against anomalib's (E00 step C2), the MPDD transfer probe (H1) and the
robustness grid (E08). Results: `reports/05`, `07`, `08`, `10`.

Every run goes through the shared runner into `reports/runs/`, keyed by a content hash, so
the notebook resumes where it stopped and never repeats a completed run. Runs on any CUDA
machine; 960 and 1280 px need more than 16 GB of host RAM. **`DRY_RUN = True`** lists the
planned runs without running them.

Data: VisA © Amazon, CC BY 4.0; MPDD, CC BY-NC-SA 4.0 (evaluation only). Neither is
redistributed; see [ATTRIBUTION.md](../ATTRIBUTION.md).

## 0. Environment

A dirty checkout tags every run dirty, and dirty runs may not enter the headline table.

In [ ]:
import os, sys, json, pathlib, subprocess

REPO = pathlib.Path.cwd()
if not (REPO / 'src' / 'inspector').exists():
    REPO = REPO.parent  # running from notebooks/
os.chdir(REPO)  # scripts and configs are addressed from the repository root
sys.path.insert(0, str(REPO / 'src'))

from inspector.utils.env import capture

env = capture()
print('git      :', env['git_sha'], '(dirty)' if env['dirty'] else '(clean)')
print('device   :', env['device'].get('gpu_name', 'CPU only'),
      f"{env['device'].get('gpu_total_mb', 0)} MB")

def sh(*args, run=True):
    """Run a repository script (or `-m inspector ...`) with this interpreter.
    The command is always printed; with run=False it is only printed."""
    cmd = [sys.executable, *map(str, args)]
    print('$ python', ' '.join(cmd[1:]), '' if run else '   (not run)')
    if run:
        subprocess.run(cmd, check=True)

In [ ]:
VISA_ROOT = pathlib.Path(os.environ.get('MVI_DATA_ROOT', 'data/raw/VisA_20220922'))
MPDD_ROOT = pathlib.Path(os.environ.get('MVI_MPDD_ROOT', 'data/raw/MPDD'))
DRY_RUN = True   # True: list the plan only; False: run (hours of GPU time)

assert (VISA_ROOT / 'split_csv' / '1cls.csv').exists(), (
    'VisA not found: notebook 01 section 1, or `python -m inspector fetch visa --out data/raw`')

def k1(*groups, extra=()):
    sh('scripts/visa_k1_experiments.py', '--data-root', VISA_ROOT, '--groups', *groups, *extra,
       *(['--dry-run'] if DRY_RUN else []))

## 1. Reference: the frozen recipe, three seeds

PatchCore, WideResNet50-2 `layer2+3`, 640 px long side, memory bank fixed at 10,000 entries
(selected in E04, frozen by ADR-11). Every comparison below is against these runs.

In [ ]:
k1('reference')

## 2. Ablation screen (K1): one axis at a time, seed 0

| axis | variants |
|---|---|
| backbone | resnet18, resnet50 |
| layers | `layer2`, `layer3`, `layer2+3+4` |
| bank size | 2.5k, 5k, 20k, 40k |
| neighbours | k = 3 and 9 (mean), k = 3 (k-th) |
| image-score re-weighting | off |
| random projection | 256, 512 dims |
| resolution | 960, 1280 px |

A screen, not an adoption rule: a variant that helps here goes to the follow-up.

In [ ]:
k1('sweep')

## 3. Gate E follow-up (E8): the screening winners at three seeds

`layer2`, 960 px, `layer2` at 960 px, and timm's `racm_in1k` weights. The decision rule DR-1 was
written down and pushed before these runs; section 8 applies it.

In [ ]:
k1('followup')

## 4. Tier 1: the convolutional autoencoder at 640 px

L2, SSIM and L2+SSIM losses, up to 150 epochs with early stopping: is the reconstruction
rung budget-limited? (It was not: it stays 0.17-0.67 image AUROC behind PatchCore.)

In [ ]:
k1('cae')

## 5. E00 step C2: our PatchCore against anomalib v2.3.0, 320 px, three seeds

Three arms: ours (torchvision weights), anomalib with its default timm weights, and anomalib
with torchvision weights. The pairs separate the pipeline from the weights.

In [ ]:
k1('c2')

## 6. MPDD transfer probe (H1, ADR-12)

The frozen recipe, unchanged, on six painted metal parts from another camera, factory and
annotation team, against the pixel-PCA and colour-histogram floors. MPDD is not
redistributed: fetch it with `python -I scripts/data_fetch_mpdd.py` first.

In [ ]:
if (MPDD_ROOT / 'bracket_black' / 'train' / 'good').is_dir():
    k1('mpdd', extra=('--mpdd-root', MPDD_ROOT))
else:
    print('MPDD not found at', MPDD_ROOT, '- run: python -I scripts/data_fetch_mpdd.py')

## 7. Robustness (E08)

Seven corruptions (plus translation) at five severities, scaled by each category's median
defect diameter; the operating threshold stays frozen on clean validation normals. A rerun
writes to `reports/studies/e08-rerun/`, never over the committed table.

In [ ]:
sh('scripts/visa_e08_robustness.py', '--data-root', VISA_ROOT,
   '--out', 'reports/studies/e08-rerun/robustness.csv', run=not DRY_RUN)

## 8. Registry table and analysis

The table is regenerated from every run in the registry. The analysis pairs each variant with
the reference on the same test images (class-stratified bootstrap, Holm over the family) and
applies DR-1 to the follow-up. Committed results: `reports/studies/kaggle/` and
`reports/studies/kaggle_e8/`; a rerun writes to `reports/studies/k1-rerun/`.

In [ ]:
from inspector.results import write_results
from inspector.runner import Registry, implementation_id

results = Registry(REPO / 'reports' / 'runs').results()
print(len(results), 'runs in the registry')
if not DRY_RUN:
    write_results(results, REPO / 'reports' / 'results_registry.csv')
sh('scripts/visa_k1_analysis.py', '--implementation', implementation_id()[:12],
   '--out', 'reports/studies/k1-rerun', run=not DRY_RUN)